In [6]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from math import sqrt
from sklearn.metrics import mean_squared_error,mean_absolute_error
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ---- Make repo importable (so `configs...` works) ----
repo_root = Path.cwd().resolve()
while repo_root.name != "graphCeylon-API" and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from configs.recommender_service.config import WEATHER_RAW_DIR, DATA_CLEAN_DIR

HISTORY_FILE = WEATHER_RAW_DIR / "cleaned_history.csv"
OUT_FORECAST = DATA_CLEAN_DIR / "sarimax_rainfall_forecast.csv"
OUT_SCORES   = DATA_CLEAN_DIR / "sarimax_rainfall_model_scores.csv"

print("History:", HISTORY_FILE)
print("Exists? ", HISTORY_FILE.exists())


History: C:\Users\HP\OneDrive\Documents\graphCeylon-API\data\recommender_service\raw\data_raw\weather\cleaned_history.csv
Exists?  True


In [7]:
if not HISTORY_FILE.exists():
    raise FileNotFoundError(f"History file not found: {HISTORY_FILE}")

df = pd.read_csv(HISTORY_FILE)
df["date"] = pd.to_datetime(df["date"])

# Normalize location_id to reduce duplicates like "adam's_peak" vs "adams_peak"
df["location_id"] = (
    df["location_id"].astype(str)
      .str.lower()
      .str.strip()
      .str.replace("’", "'", regex=False)
      .str.replace("'", "", regex=False)         # remove apostrophes
      .str.replace("-", "_", regex=False)
      .str.replace(" ", "_", regex=False)
)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "location_id", "rainfall"]).copy()

df["rainfall"] = pd.to_numeric(df["rainfall"], errors="coerce").fillna(0.0).clip(lower=0)
if "avg_temp" in df.columns:
    df["avg_temp"] = pd.to_numeric(df["avg_temp"], errors="coerce")



# Optional: keep avg_temp as exog if available
if "avg_temp" in df.columns:
    df["avg_temp"] = pd.to_numeric(df["avg_temp"], errors="coerce")

print(df.head())
print("Locations:", df["location_id"].nunique())


        date location_id  avg_temp  rainfall
0 2021-01-01  adams_peak     15.55      91.4
1 2021-02-01  adams_peak     15.40      59.2
2 2021-03-01  adams_peak     16.15     108.7
3 2021-04-01  adams_peak     16.60     107.2
4 2021-05-01  adams_peak     17.20     273.2
Locations: 38


In [ ]:
# Try both parses and pick the one where most rows fall on day=1 (monthly series)
d0 = pd.to_datetime(df["date"], errors="coerce", dayfirst=False)
d1 = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)

score0 = (d0.dt.day == 1).mean()
score1 = (d1.dt.day == 1).mean()

df["date"] = d0 if score0 >= score1 else d1
print("Picked dayfirst =", (score1 > score0), " | day==1 ratio:", max(score0, score1))


Picked dayfirst = False  | day==1 ratio: 1.0


In [ ]:
def ensure_monthly(loc_df: pd.DataFrame) -> pd.DataFrame:
    loc_df = loc_df.sort_values("date").copy()

    loc_df["date"] = loc_df["date"].dt.to_period("M").dt.to_timestamp("MS")

    agg = {"rainfall": "sum"}  # rainfall must be monthly total
    if "avg_temp" in loc_df.columns:
        agg["avg_temp"] = "mean"

    loc_df = (
        loc_df.groupby(["location_id", "date"], as_index=False)
              .agg(agg)
              .sort_values("date")
    )

    start = loc_df["date"].min()
    end   = loc_df["date"].max()
    idx = pd.date_range(start, end, freq="MS")

    loc_df = loc_df.set_index("date").reindex(idx)
    loc_df.index.name = "date"

    loc_df["location_id"] = loc_df["location_id"].ffill().bfill()
    loc_df["rainfall"] = loc_df["rainfall"].fillna(0.0).clip(lower=0)

    if "avg_temp" in loc_df.columns:
        loc_df["avg_temp"] = (
            loc_df["avg_temp"]
            .interpolate(limit_direction="both")
            .bfill()
            .ffill()
        )

    return loc_df.reset_index()



def make_exog(dates: pd.Series, avg_temp: pd.Series | None = None) -> pd.DataFrame:
    m = dates.dt.month
    exog = pd.DataFrame({
        "month_sin": np.sin(2*np.pi*m/12),
        "month_cos": np.cos(2*np.pi*m/12),
    }, index=dates.index)

    if avg_temp is not None:
        exog["avg_temp"] = avg_temp.values

    return exog


def transform_rain(y_mm, cap_q=0.98):
    y = pd.Series(y_mm).astype(float).clip(lower=0)
    cap = y.quantile(cap_q)
    y_cap = y.clip(upper=cap)
    return np.log1p(y_cap), cap

def inv_transform_rain(y_log, cap):
    y_mm = np.expm1(y_log)
    return np.clip(y_mm, 0, cap)

def transform_rain(y_mm, cap_q=0.98):
    y = pd.Series(y_mm).astype(float).clip(lower=0)
    cap = y.quantile(cap_q)
    y_cap = y.clip(upper=cap)
    return np.log1p(y_cap), cap

def inv_transform_rain(y_log, cap):
    y_mm = np.expm1(y_log)
    return np.clip(y_mm, 0, cap)



In [ ]:
# Small but effective grid for monthly rainfall
P = [0, 1, 2]
D = [0, 1]          # allow differencing
Q = [0, 1, 2]

SP = [0, 1]
SD = [0, 1]
SQ = [0, 1]
S  = 12             # monthly seasonality

def rmse_mm(y_true_mm, y_pred_mm):
    return np.sqrt(mean_squared_error(y_true_mm, y_pred_mm))

def smape(y_true, y_pred, eps=1e-6):
    denom = (np.abs(y_true) + np.abs(y_pred) + eps)
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)

def peak_rmse(y_true, y_pred, k=3):
    idx = np.argsort(y_true)[-k:]
    return np.sqrt(mean_squared_error(y_true[idx], y_pred[idx]))

def fit_eval_sarimax(loc_df: pd.DataFrame, test_horizon=12):
    loc_df = loc_df.sort_values("date").copy()

    if len(loc_df) < (test_horizon + 18):
        test_horizon = max(6, len(loc_df) // 4)

    cut = len(loc_df) - test_horizon
    train_df = loc_df.iloc[:cut].copy()
    test_df  = loc_df.iloc[cut:].copy()

    # 🔑 Stable rainfall transform
    train_y_log, cap = transform_rain(train_df["rainfall"].values)
    test_y_mm = test_df["rainfall"].values

    use_temp = "avg_temp" in loc_df.columns
    train_exog = make_exog(train_df["date"], train_df["avg_temp"] if use_temp else None)
    test_exog  = make_exog(test_df["date"],  test_df["avg_temp"] if use_temp else None)

    best = None  # (rmse, res, cfg, metrics)

    for p in P:
        for d in D:
            for q in Q:
                for sp in SP:
                    for sd in SD:
                        for sq in SQ:
                            order = (p, d, q)
                            seasonal_order = (sp, sd, sq, S)

                            try:
                                model = SARIMAX(
                                    train_y_log,
                                    exog=train_exog,
                                    order=order,
                                    seasonal_order=seasonal_order,
                                    enforce_stationarity=False,
                                    enforce_invertibility=False,
                                )
                                res = model.fit(disp=False, maxiter=300)

                                pred_log = res.get_forecast(
                                    steps=len(test_df),
                                    exog=test_exog
                                ).predicted_mean

                                pred_mm = inv_transform_rain(pred_log, cap)

                                rmse = rmse_mm(test_y_mm, pred_mm)
                                mae  = mean_absolute_error(test_y_mm, pred_mm)
                                sm   = smape(test_y_mm, pred_mm)
                                prmse = peak_rmse(test_y_mm, pred_mm)

                                if not np.isfinite(rmse):
                                    continue

                                metrics = {
                                    "rmse": rmse,
                                    "mae": mae,
                                    "smape": sm,
                                    "peak_rmse": prmse,
                                    "aic": res.aic,
                                    "bic": res.bic,
                                    "converged": bool(res.mle_retvals.get("converged", True))
                                }

                                if best is None or rmse < best[0]:
                                    best = (
                                        rmse,
                                        res,
                                        {
                                            "order": order,
                                            "seasonal_order": seasonal_order,
                                            "use_temp": use_temp,
                                            "cap": cap
                                        },
                                        metrics
                                    )
                            except Exception:
                                continue

    if best is None:
        return None, None, None, None, None

    best_res, best_cfg, metrics, train_df, test_df = fit_eval_sarimax(loc_df, test_horizon=test_horizon)
    best_rmse = metrics["rmse"]






In [ ]:
forecast_horizon = 24
test_horizon = 12

all_fc = []
scores = []

locations = sorted(df["location_id"].unique())

for loc in locations:
    loc_df = df[df["location_id"] == loc].copy()
    loc_df = ensure_monthly(loc_df)

    # skip tiny
    if len(loc_df) < 18:
        print(f"⚠️ Skipping {loc} (too little data: {len(loc_df)})")
        continue

    print(f"📍 SARIMAX: {loc} ({len(loc_df)} months)")

    best_res, best_cfg, best_rmse, train_df, test_df = fit_eval_sarimax(loc_df, test_horizon=test_horizon)

    if best_res is None:
        print(f"   ❌ No model converged for {loc}")
        continue

    # Refit on full data using best cfg
    full_y = np.log1p(loc_df["rainfall"].values)

    use_temp = best_cfg["use_temp"]
    full_exog = make_exog(loc_df["date"], loc_df["avg_temp"] if (use_temp and "avg_temp" in loc_df.columns) else None)

    refit = SARIMAX(
        full_y,
        exog=full_exog,
        order=best_cfg["order"],
        seasonal_order=best_cfg["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)

    last_date = loc_df["date"].max()
    future_dates = pd.date_range(last_date + pd.offsets.MonthBegin(1), periods=forecast_horizon, freq="MS")

    # future exog (we may not have future temp; if not, we just omit temp feature)
    if use_temp:
        # simplest: use climatology = monthly mean temp for that location
        # (later we can replace with your temp forecast)
        temp_monthly_mean = loc_df.groupby(loc_df["date"].dt.month)["avg_temp"].mean()
        future_temp = pd.Series([temp_monthly_mean[d.month] for d in future_dates])
        future_exog = make_exog(pd.Series(future_dates), future_temp)
    else:
        future_exog = make_exog(pd.Series(future_dates), None)

    pred_log = refit.get_forecast(steps=forecast_horizon, exog=future_exog).predicted_mean
    pred_mm = np.expm1(pred_log).clip(min=0)

    fc = pd.DataFrame({
        "ds": future_dates,
        "location_id": loc,
        "pred_rainfall": np.round(pred_mm.values, 1)
    })
    all_fc.append(fc)

    scores.append({
    "location_id": loc,
    "rmse_mm_last12": round(best_rmse, 2),
    "mae_mm": round(metrics["mae"], 2),
    "smape_pct": round(metrics["smape"], 2),
    "peak_rmse_mm": round(metrics["peak_rmse"], 2),
    "order": str(best_cfg["order"]),
    "seasonal_order": str(best_cfg["seasonal_order"]),
    "used_temp_exog": bool(use_temp),
    "converged": metrics["converged"],
    "n_months": len(loc_df)
    })


# Save
DATA_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

if all_fc:
    final_fc = pd.concat(all_fc, ignore_index=True)
    final_fc.to_csv(OUT_FORECAST, index=False)
    print(f"\n✅ Saved forecast -> {OUT_FORECAST}")

scores_df = pd.DataFrame(scores).sort_values("rmse_mm_last12", ascending=True)
scores_df.to_csv(OUT_SCORES, index=False)
print(f"✅ Saved scores   -> {OUT_SCORES}")

scores_df.head(15)


📍 SARIMAX: adams_peak (60 months)


c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607:

   ❌ No model converged for adams_peak
📍 SARIMAX: anuradhapura (60 months)


c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607:

   ❌ No model converged for anuradhapura
📍 SARIMAX: arugambay (53 months)


...

c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607:

   ❌ No model converged for ella
📍 SARIMAX: galle (60 months)


c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607:

   ❌ No model converged for galle
📍 SARIMAX: hambantota (60 months)


c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: